# Exploratory Data Analysis

Understanding the dataset to explore how the data is present in the database and determine whether there is a need to create aggregated tables that can help with:

- Vendor selection for profitability
- Product pricing optimization

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
conn = sqlite3.connect('inventory.db')

In [ ]:
tables = pd.read_sql_query("SELECT name from sqlite_master WHERE type='table'",conn)
tables


In [ ]:
for table in tables['name']:
    print(f"\n{'-'*50} {table} {'-'*50}")
    print('Count of Records : ',pd.read_sql(f"select count(*) as count from {table}",conn)['count'].values[0])
    
    display(pd.read_sql(f"select * from {table} limit 5",conn))

#### Analysing One Venddor 4166 to see how data is stored

In [ ]:
def show(query):
    return pd.read_sql_query(query, conn)

query = "select * from purchases where VendorNumber = 4466 limit 5"

show(query)

In [ ]:
def show(query):
    return pd.read_sql_query(query, conn)

query = "select * from purchase_prices where VendorNumber = 4466 "

show(query)

In [ ]:
def show(query):
    return pd.read_sql_query(query, conn)

query = "select * from vendor_invoice where VendorNumber = 4466 "

show(query)

In [ ]:
def show(query):
    return pd.read_sql_query(query, conn)

query = "select * from sales where VendorNo=4466"

show(query)

In [ ]:
query = "select * from purchases where VendorNumber = 4466"
purchases = show(query)
purchases

In [ ]:
query = "select * from purchase_prices where VendorNumber = 4466"
purchase_prices = show(query)
purchase_prices

In [ ]:
query = "select * from sales where VendorNo = 4466"
sales = show(query)
sales

In [ ]:
purchases.groupby(['Brand','PurchasePrice'])[['Quantity','Dollars']].sum()

In [ ]:
purchase_prices

In [ ]:
query = "select * from vendor_invoice where VendorNumber=4466"
vendor_invoice = show(query)
vendor_invoice

In [ ]:
vendor_invoice.columns

In [ ]:
purchase_prices.columns

In [ ]:
purchases.columns

In [ ]:
sales.columns

In [ ]:
sales

In [ ]:
sales.groupby(['Brand','SalesPrice'])[['SalesDollars','SalesQuantity']].sum()

In [ ]:
purchases.groupby(['Brand','PurchasePrice'])[['Quantity','Dollars']].sum()

- The purchases table contains actual purchase data, including the date of purchase, products (brands) purchased by vendors, the amount paid (in dollars), and the quantity purchased.

- The purchase price column is derived from the purchase_prices table, which provides product-wise actual and purchase prices. The combination of vendor and brand is unique in this table.

- The vendor_invoice table aggregates data from the purchases table, summarizing quantity and dollar amounts, along with an additional column for freight. This table maintains uniqueness based on vendor and PO number.

- The sales table captures actual sales transactions, detailing the brands purchased by vendors, the quantity sold, the selling price, and the revenue earned.

As the data that we need for analysis is distributed in different tables, we need to create a summary table containing:

- purchase transactions made by vendors
- sales transaction data
- freight costs for each vendor
- actual product prices from vendors

In [ ]:
vendor_invoice.columns

In [ ]:
#group by on vendor number freight we sum 
query = """
select VendorNumber, VendorName,SUM(Freight) as FreightCost
from vendor_invoice
group by VendorNumber
"""
freight_summary = show(query)
freight_summary = freight_summary[
    ["VendorNumber", "VendorName", "FreightCost"]
]


#Total freight costs per vendor 
#freight_summary[freight_summary['VendorNumber']==4466]


In [ ]:
freight_summary

In [ ]:
# join purchase prices and purchases table to see the difference 

In [ ]:
purchase_prices.columns

In [ ]:
purchases.columns

In [ ]:
query = """
SELECT
    p.VendorNumber,
    p.VendorName,
    p.Brand,
    SUM(p.Quantity) AS TotalPurchaseQuantity,
    SUM(p.Dollars) AS TotalPurchaseDollars,
    pp.Volume,
    pp.Price AS ActualPrice
FROM purchases p
JOIN purchase_prices pp
    ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
GROUP BY
    p.VendorNumber,
    p.VendorName,
    p.Brand,
    pp.Volume,
    pp.Price
ORDER BY
    p.VendorNumber

"""

show(query)

In [ ]:
sales.columns

In [ ]:
query = """ 
SELECT
    VendorNo,
    Brand,
    SUM(SalesDollars) as TotalSalesDollars,
    SUM(SalesPrice) as TotalSalesPrice,
    SUM(SalesQuantity) as TotalSalesQuantity,
    SUM(ExciseTax) as TotalExciseTax
FROM 
    sales
GROUP BY
    VendorNo,
    Brand
ORDER BY 
    TotalSalesDollars
"""

show(query)

In [ ]:
vendor_invoice.head(5)

In [ ]:
'''
import time 
start = time.time()

query = """
SELECT
    pp.VendorNumber,
    pp.Brand,
    pp.Price as ActualPrice,
    pp.PurchasePrice as PurchasePrice,
    SUM(s.SalesQuantity) AS TotalSalesQuantity,
    SUM(s.SalesDollars) AS TotalSalesDollars,
    SUM(s.SalesPrice) AS TotalSalesPrice,
    SUM(s.ExciseTax) AS TotalExciseTax,
    SUM(v.Quantity) AS TotalPurchaseQuantity,
    SUM(v.Dollars) AS TotalPurchaseDollars,
    SUM(v.Freight) AS Freight
FROM purchase_prices pp
JOIN sales s
    ON pp.VendorNumber = s.VendorNo
    AND pp.Brand = s.Brand
JOIN vendor_invoice v
    on pp.VendorNumber = v.VendorNumber
GROUP BY
    pp.VendorNumber,
    pp.Brand,
    pp.Price,
    pp.PurchasePrice
"""
final_table = show(query)
end = time.time()
print(f"Time required is {(end-start)/60:2f}")

'''
print('Skipped - OPtimised Query Written Next')

In [ ]:
query = """
WITH FreightSummary AS (
    SELECT
        VendorNumber,
        SUM(Freight) AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
),

PurchaseSummary AS (
    SELECT
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        p.Description,
        p.PurchasePrice,
        pp.Price AS ActualPrice,
        pp.Volume,
        SUM(p.Quantity) AS TotalPurchaseQuantity,
        SUM(p.Dollars) AS TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp
        ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        p.Description,
        p.PurchasePrice,
        pp.Price,
        pp.Volume
),

SalesSummary AS (
    SELECT
        VendorNo,
        Brand,
        SUM(SalesQuantity) AS TotalSalesQuantity,
        SUM(SalesDollars) AS TotalSalesDollars,
        SUM(SalesPrice) AS TotalSalesPrice,
        SUM(ExciseTax) AS TotalExciseTax
    FROM sales
    GROUP BY
        VendorNo,
        Brand
)

SELECT
    ps.VendorNumber,
    ps.VendorName,
    ps.Brand,
    ps.Description,
    ps.PurchasePrice,
    ps.ActualPrice,
    ps.Volume,
    ps.TotalPurchaseQuantity,
    ps.TotalPurchaseDollars,
    ss.TotalSalesQuantity,
    ss.TotalSalesDollars,
    ss.TotalSalesPrice,
    ss.TotalExciseTax,
    fs.FreightCost
FROM PurchaseSummary ps
LEFT JOIN SalesSummary ss
    ON ps.VendorNumber = ss.VendorNo
   AND ps.Brand = ss.Brand
LEFT JOIN FreightSummary fs
    ON ps.VendorNumber = fs.VendorNumber
ORDER BY
    ps.TotalPurchaseDollars DESC
"""

vendor_sales_summary = pd.read_sql_query(query, conn)

In [ ]:
vendor_sales_summary

This query generates a vendor-wise sales and purchase summary, which is valuable for:

### Performance Optimization:

- The query involves heavy joins and aggregations on large datasets like sales and purchases.
- Storing the pre-aggregated results avoids repeated expensive computations.
- Helps in analyzing sales, purchases, and pricing for different vendors and brands.
- Future benefits of storing this data for faster dashboarding & reporting.
- Instead of running expensive queries each time, dashboards can fetch data quickly from `vendor_sales_summary`.

In [ ]:
vendor_sales_summary.info()

In [ ]:
vendor_sales_summary.isnull().sum()

In [ ]:
vendor_sales_summary['VendorName'].unique()

In [ ]:
final_table['Description'].unique

In [ ]:
vendor_sales_summary['Volume']=vendor_sales_summary['Volume'].astype('float64')

In [ ]:
vendor_sales_summary.fillna(0)

In [ ]:
vendor_sales_summary['VendorName']=vendor_sales_summary['VendorName'].str.strip()

In [ ]:
vendor_sales_summary.head(5)

In [ ]:
vendor_sales_summary['GrossProfit'] = vendor_sales_summary['TotalSalesDollars']-vendor_sales_summary['TotalPurchaseDollars']

In [ ]:
vendor_sales_summary['ProfitMargin'] = round((vendor_sales_summary['GrossProfit']/vendor_sales_summary['TotalSalesDollars'])*100,2)

In [ ]:
vendor_sales_summary['StockTurnover'] = round((vendor_sales_summary['TotalSalesQuantity']/vendor_sales_summary['TotalPurchaseQuantity']),2)

In [ ]:
vendor_sales_summary['SalesToPurchaseRatio'] = round((vendor_sales_summary['TotalSalesDollars']/vendor_sales_summary['TotalPurchaseDollars']),2)

In [ ]:
vendor_sales_summary['NetProfit'] = vendor_sales_summary['TotalSalesDollars']-vendor_sales_summary['TotalPurchaseDollars']-vendor_sales_summary['FreightCost']

In [ ]:
vendor_sales_summary["AvgBuyPrice"] = (
    vendor_sales_summary["TotalPurchaseDollars"]
    / vendor_sales_summary["TotalPurchaseQuantity"]
)

vendor_sales_summary["AvgSellPrice"] = (
    vendor_sales_summary["TotalSalesDollars"]
    / vendor_sales_summary["TotalSalesQuantity"]
)

vendor_sales_summary["PriceDifference"] = (
    vendor_sales_summary["AvgSellPrice"]
    - vendor_sales_summary["AvgBuyPrice"]
)

vendor_sales_summary["PriceDifferencePct"] = (
    vendor_sales_summary["PriceDifference"]
    / vendor_sales_summary["AvgBuyPrice"]
) * 100

In [ ]:
vendor_sales_summary.info()

In [ ]:
vendor_sales_summary.columns

In [ ]:
cursor = conn.cursor()

In [ ]:
create_query = """ 
CREATE TABLE vendor_sales_summary (
    VendorNumber INT,
    VendorName VARCHAR(100),
    Brand INT,
    Description VARCHAR(255),

    PurchasePrice DECIMAL(15,2),
    ActualPrice DECIMAL(15,2),
    Volume DECIMAL(15,2),

    TotalPurchaseQuantity INT,
    TotalPurchaseDollars DECIMAL(15,2),

    TotalSalesQuantity DECIMAL(15,2),
    TotalSalesDollars DECIMAL(15,2),
    TotalSalesPrice DECIMAL(15,2),
    TotalExciseTax DECIMAL(15,2),

    FreightCost DECIMAL(15,2),

    GrossProfit DECIMAL(15,2),
    ProfitMargin DECIMAL(15,2),

    StockTurnover DECIMAL(15,2),
    SalesToPurchaseRatio DECIMAL(15,2),

    NetProfit DECIMAL(15,2),

    AvgBuyPrice DECIMAL(15,2),
    AvgSellPrice DECIMAL(15,2),

    PriceDifference DECIMAL(15,2),
    PriceDifferencePct DECIMAL(15,2),

    PRIMARY KEY (VendorNumber, Brand)
);
"""

In [ ]:
cursor.execute(create_query)

In [ ]:
#Check Table Creation 
show("SELECT * FROM vendor_sales_summary")

In [ ]:
vendor_sales_summary.to_sql('vendor_sales_summary',conn,if_exists='replace',index=False)